# Methodology — Quy trình thu thập dữ liệu

## 1. Discovery và version resolution

Đầu vào là danh sách 5.000 arXiv IDs được giao cho Member 5. Discovery truy vấn arXiv API để lấy metadata và xác định toàn bộ version của mỗi paper. Mỗi version được chuyển thành một `version_task` riêng để bảo đảm source crawling không bỏ qua các lần sửa đổi.

## 2. Source crawling và extraction

Source archive được tải theo version từ endpoint source của arXiv. Theo yêu cầu M1, pipeline chỉ giữ `.tex` và `.bib`; figure binaries và các tệp không thuộc phạm vi không được đưa vào corpus. Các lệnh LaTeX tham chiếu hình trong `.tex` được giữ nguyên vì Milestone 2 cần phân tích cấu trúc tài liệu.

Các version không có TeX/Bib vẫn được giữ trong manifest với trạng thái `no_tex_bib`. Các lỗi HTTP và archive không hợp lệ cũng được ghi thành trạng thái riêng, không bị loại khỏi thống kê.

## 3. Reference crawling

Semantic Scholar được truy vấn theo từng paper. Tất cả references crawl được lưu trong dữ liệu làm việc. Chỉ references có arXiv ID hợp lệ mới được đưa vào bảng `references` của M1. References không có arXiv ID vẫn được giữ lại cục bộ để không làm mất dữ liệu crawl.

## 4. Build và validation

Sau khi crawl kết thúc, pipeline tạo hai bảng Parquet `papers` và `references`. Validation đối chiếu danh sách paper được giao, kiểm tra số dòng và tính duy nhất, kiểm tra references, đồng thời xác nhận mọi version task có đúng một terminal status.

## 5. Công cụ

Các stage được gọi qua CLI Python của dự án (`src.cli`) trong virtual environment. Các stage đã chạy trên shard Member 5 gồm `discover`, `crawl-sources`, `crawl-references`, `build-tables` và `validate`. Dữ liệu làm việc được lưu ngoài repository mã nguồn.


## 6. Mô hình trạng thái

`success` biểu thị stage đã tạo output hợp lệ; `no_tex_bib` biểu thị version không có TeX/Bib phù hợp; `http_error` và `invalid_archive` biểu thị các vấn đề source retrieval. Với Semantic Scholar, `ok` và `not_found` là hai trạng thái kết thúc quan sát được trong shard này. Các trạng thái phải được giữ nguyên trong báo cáo và không được sửa thủ công để làm đẹp tỷ lệ thành công.

## 7. Khả năng tái lập

Khi bàn giao, cần lưu `member_id`, commit SHA của code, danh sách input/checksum, lệnh thực thi, thời điểm bắt đầu/kết thúc, phiên bản Python và package, cùng các manifest/summary/validation artifacts. API key nếu sử dụng chỉ được lưu trong biến môi trường hoặc secret store, không đưa vào Git.
